<a href="https://colab.research.google.com/github/geethat0126-bit/AI_5SEM/blob/main/week9_1BM24CS104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Expr: pass

class Var(Expr):
    def __init__(self, name): self.name = name
    def __str__(self): return self.name

class Func(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __str__(self): return f"{self.name}({', '.join(map(str, self.args))})" if self.args else self.name

class Pred(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __str__(self): return f"{self.name}({', '.join(map(str, self.args))})"

class Not(Expr):
    def __init__(self, op): self.op = op
    def __str__(self): return f"¬{self.op}" if isinstance(self.op, (Pred, Not)) else f"¬({self.op})"

class And(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ⇒ {self.right})"

class Quant(Expr):
    def __init__(self, q, v, b): self.qtype, self.var, self.body = q, v, b
    def __str__(self): return f"{'∀' if self.qtype == 'FORALL' else '∃'}{self.var} [{self.body}]"

def tokenize(text):
    for k, v in {"∀":" forall ","∃":" exists ","⇒":" => ","->":" => ","∧":" & ","∨":" | ","¬":" ~ ","~":" ~ ","[":" ( ","]":" ) ","(":" ( ",")":" ) ",",":" , "}.items():
        text = text.replace(k, v)
    return text.split()

class Parser:
    def __init__(self, tokens): self.tokens, self.pos = tokens, 0
    def peek(self): return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    def consume(self, exp=None):
        tok = self.peek()
        if not tok or (exp and tok != exp): raise ValueError(f"Expected {exp}, got {tok}")
        self.pos += 1
        return tok
    def parse(self):
        e = self.parse_implies()
        if self.peek(): raise ValueError(f"Unexpected token: {self.peek()}")
        return e
    def parse_implies(self):
        l = self.parse_or()
        while self.peek() == "=>": self.consume("=>"); l = Implies(l, self.parse_or())
        return l
    def parse_or(self):
        l = self.parse_and()
        while self.peek() == "|": self.consume("|"); l = Or(l, self.parse_and())
        return l
    def parse_and(self):
        l = self.parse_unary()
        while self.peek() == "&": self.consume("&"); l = And(l, self.parse_unary())
        return l
    def parse_unary(self):
        t = self.peek()
        if t == "~": self.consume("~"); return Not(self.parse_unary())
        if t in ("forall", "exists"): return Quant("FORALL" if self.consume() == "forall" else "EXISTS", self.consume(), self.parse_unary())
        if t == "(": self.consume("("); e = self.parse_implies(); self.consume(")"); return e
        name = self.consume(); self.consume("("); args = []
        if self.peek() != ")":
            args.append(self.parse_term())
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
        self.consume(")")
        return Pred(name, args)
    def parse_term(self):
        name = self.consume()
        if self.peek() == "(":
            self.consume("("); args = [self.parse_term()]
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
            self.consume(")")
            return Func(name, args)
        return Var(name) if name.islower() else Func(name, [])

def elim_imp(e):
    if isinstance(e, Implies): return Or(Not(elim_imp(e.left)), elim_imp(e.right))
    if isinstance(e, Not): return Not(elim_imp(e.op))
    if isinstance(e, And): return And(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Or): return Or(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, elim_imp(e.body))
    return e

def move_not(e):
    if isinstance(e, Not):
        i = e.op
        if isinstance(i, Not): return move_not(i.op)
        if isinstance(i, And): return Or(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Or): return And(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Quant): return Quant("EXISTS" if i.qtype == "FORALL" else "FORALL", i.var, move_not(Not(i.body)))
        return Not(move_not(i))
    if isinstance(e, And): return And(move_not(e.left), move_not(e.right))
    if isinstance(e, Or): return Or(move_not(e.left), move_not(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, move_not(e.body))
    return e

def std_vars(e, used=None, m=None):
    used, m = used or set(), m or {}
    if isinstance(e, Var): return Var(m.get(e.name, e.name))
    if isinstance(e, Func): return Func(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Not): return Not(std_vars(e.op, used, m))
    if isinstance(e, And): return And(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Or): return Or(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Quant):
        nv = e.var
        while nv in used: nv = chr(ord(nv) + 1) if nv < 'z' else nv + "_1"
        used.add(nv); nm = dict(m); nm[e.var] = nv
        return Quant(e.qtype, nv, std_vars(e.body, used, nm))
    return e

sk_count = 0
def subst(e, v, rep):
    if isinstance(e, Var): return rep if e.name == v else e
    if isinstance(e, Func): return Func(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Not): return Not(subst(e.op, v, rep))
    if isinstance(e, And): return And(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Or): return Or(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, subst(e.body, v, rep))
    return e

def skolemize(e, uvars=None):
    global sk_count
    uvars = uvars or []
    if isinstance(e, Quant):
        if e.qtype == "FORALL": return Quant("FORALL", e.var, skolemize(e.body, uvars + [e.var]))
        sk = chr(ord('F') + sk_count); sk_count += 1
        return skolemize(subst(e.body, e.var, Func(sk, [Var(v) for v in uvars])), uvars)
    if isinstance(e, Not): return Not(skolemize(e.op, uvars))
    if isinstance(e, And): return And(skolemize(e.left, uvars), skolemize(e.right, uvars))
    if isinstance(e, Or): return Or(skolemize(e.left, uvars), skolemize(e.right, uvars))
    return e

def drop_univ(e):
    if isinstance(e, Quant): return drop_univ(e.body)
    if isinstance(e, Not): return Not(drop_univ(e.op))
    if isinstance(e, And): return And(drop_univ(e.left), drop_univ(e.right))
    if isinstance(e, Or): return Or(drop_univ(e.left), drop_univ(e.right))
    return e

def dist_or(e):
    if isinstance(e, And): return And(dist_or(e.left), dist_or(e.right))
    if isinstance(e, Or):
        a, b = dist_or(e.left), dist_or(e.right)
        if isinstance(a, And): return And(dist_or(Or(a.left, b)), dist_or(Or(a.right, b)))
        if isinstance(b, And): return And(dist_or(Or(a, b.left)), dist_or(Or(a, b.right)))
        return Or(a, b)
    return e

def get_clauses(e):
    return get_clauses(e.left) + get_clauses(e.right) if isinstance(e, And) else [str(e)]

if __name__ == "__main__":
    print("Symbols: forall (∀), exists (∃), => (⇒), & (∧), | (∨), ~ (¬)")
    s = input("Enter FOL Formula: ").strip()
    if s:
        e = Parser(tokenize(s)).parse()
        print("\nOriginal:", e)
        e = elim_imp(e); print("1. Eliminate implications:", e)
        e = move_not(e); print("2. Move ¬ inwards:", e)
        e = std_vars(e); print("3. Standardize variables:", e)
        e = skolemize(e); print("4. Skolemization:", e)
        e = drop_univ(e); print("5. Drop universal quantifiers:", e)
        e = dist_or(e); print("6. Distribute ∨ over ∧ (CNF):", e)
        print("\n7. Final CNF Clauses:")
        for i, c in enumerate(get_clauses(e), 1): print(f"   Clause {i}: {c}")


Symbols: forall (∀), exists (∃), => (⇒), & (∧), | (∨), ~ (¬)
Enter FOL Formula: ∀x [∀y Animal(y) ⇒ Loves(x, y)] ⇒ [∃y Loves(y, x)]

Original: (∀x [(∀y [Animal(y)] ⇒ Loves(x, y))] ⇒ ∃y [Loves(y, x)])
1. Eliminate implications: (¬(∀x [(¬(∀y [Animal(y)]) ∨ Loves(x, y))]) ∨ ∃y [Loves(y, x)])
2. Move ¬ inwards: (∃x [(∀y [Animal(y)] ∧ ¬Loves(x, y))] ∨ ∃y [Loves(y, x)])
3. Standardize variables: (∃x [(∀y [Animal(y)] ∧ ¬Loves(x, y))] ∨ ∃y [Loves(y, x)])
4. Skolemization: ((∀y [Animal(y)] ∧ ¬Loves(F, y)) ∨ Loves(G, x))
5. Drop universal quantifiers: ((Animal(y) ∧ ¬Loves(F, y)) ∨ Loves(G, x))
6. Distribute ∨ over ∧ (CNF): ((Animal(y) ∨ Loves(G, x)) ∧ (¬Loves(F, y) ∨ Loves(G, x)))

7. Final CNF Clauses:
   Clause 1: (Animal(y) ∨ Loves(G, x))
   Clause 2: (¬Loves(F, y) ∨ Loves(G, x))


In [4]:
class Expr: pass

class Var(Expr):
    def __init__(self, n): self.name = n
    def __str__(self): return self.name

class Func(Expr):
    def __init__(self, n, a): self.name, self.args = n, a
    def __str__(self): return f"{self.name}({','.join(map(str, self.args))})" if self.args else self.name

class Pred(Expr):
    def __init__(self, n, a): self.name, self.args = n, a
    def __str__(self): return f"{self.name}({','.join(map(str, self.args))})"

class Not(Expr):
    def __init__(self, o): self.op = o
    def __str__(self): return f"¬{self.op}" if isinstance(self.op, (Pred, Not)) else f"¬({self.op})"

class And(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, l, r): self.left, self.right = l, r
    def __str__(self): return f"({self.left} → {self.right})"

class Quant(Expr):
    def __init__(self, q, v, b): self.qtype, self.var, self.body = q, v, b
    def __str__(self): return f"({'∀' if self.qtype == 'FORALL' else '∃'}{self.var}){self.body}"

def tokenize(text):
    for k, v in {"∀":" forall ","∃":" exists ","→":" -> ","⇒":" -> ","^":" & ","∧":" & ","∨":" | ","¬":" ~ ","~":" ~ ","[":" ( ","]":" ) ","(":" ( ",")":" ) ",",":" , "}.items():
        text = text.replace(k, v)
    return text.split()

class Parser:
    def __init__(self, tokens): self.tokens, self.pos = tokens, 0
    def peek(self): return self.tokens[self.pos] if self.pos < len(self.tokens) else None
    def consume(self, exp=None):
        t = self.peek()
        if not t or (exp and t != exp): raise ValueError(f"Expected {exp}, got {t}")
        self.pos += 1
        return t
    def parse(self):
        e = self.parse_imp()
        if self.peek(): raise ValueError(f"Unexpected: {self.peek()}")
        return e
    def parse_imp(self):
        l = self.parse_or()
        while self.peek() == "->": self.consume("->"); l = Implies(l, self.parse_or())
        return l
    def parse_or(self):
        l = self.parse_and()
        while self.peek() == "|": self.consume("|"); l = Or(l, self.parse_and())
        return l
    def parse_and(self):
        l = self.parse_unary()
        while self.peek() == "&": self.consume("&"); l = And(l, self.parse_unary())
        return l
    def parse_unary(self):
        t = self.peek()
        if t == "~": self.consume("~"); return Not(self.parse_unary())
        if t == "(":
            if self.pos + 1 < len(self.tokens) and self.tokens[self.pos + 1] in ("forall", "exists"):
                self.consume("(")
                q = "FORALL" if self.consume() == "forall" else "EXISTS"
                v = self.consume()
                self.consume(")")
                return Quant(q, v, self.parse_unary())
            self.consume("("); e = self.parse_imp(); self.consume(")"); return e
        if t in ("forall", "exists"):
            return Quant("FORALL" if self.consume() == "forall" else "EXISTS", self.consume(), self.parse_unary())
        name = self.consume(); self.consume("("); args = []
        if self.peek() != ")":
            args.append(self.parse_term())
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
        self.consume(")")
        return Pred(name, args)
    def parse_term(self):
        name = self.consume()
        if self.peek() == "(":
            self.consume("("); args = [self.parse_term()]
            while self.peek() == ",": self.consume(","); args.append(self.parse_term())
            self.consume(")")
            return Func(name, args)
        return Var(name) if name.islower() else Func(name, [])

def elim_imp(e):
    if isinstance(e, Implies): return Or(Not(elim_imp(e.left)), elim_imp(e.right))
    if isinstance(e, Not): return Not(elim_imp(e.op))
    if isinstance(e, And): return And(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Or): return Or(elim_imp(e.left), elim_imp(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, elim_imp(e.body))
    return e

def move_not(e):
    if isinstance(e, Not):
        i = e.op
        if isinstance(i, Not): return move_not(i.op)
        if isinstance(i, And): return Or(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Or): return And(move_not(Not(i.left)), move_not(Not(i.right)))
        if isinstance(i, Quant): return Quant("EXISTS" if i.qtype == "FORALL" else "FORALL", i.var, move_not(Not(i.body)))
        return Not(move_not(i))
    if isinstance(e, And): return And(move_not(e.left), move_not(e.right))
    if isinstance(e, Or): return Or(move_not(e.left), move_not(e.right))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, move_not(e.body))
    return e

def std_vars(e, used=None, m=None):
    used, m = used if used is not None else set(), m if m is not None else {}
    if isinstance(e, Var): return Var(m.get(e.name, e.name))
    if isinstance(e, Func): return Func(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [std_vars(a, used, m) for a in e.args])
    if isinstance(e, Not): return Not(std_vars(e.op, used, m))
    if isinstance(e, And): return And(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Or): return Or(std_vars(e.left, used, m), std_vars(e.right, used, m))
    if isinstance(e, Quant):
        nv = e.var
        while nv in used: nv = 'z' if nv == 'x' else ('w' if nv == 'y' else chr(ord(nv) + 1))
        used.add(nv); nm = dict(m); nm[e.var] = nv
        return Quant(e.qtype, nv, std_vars(e.body, used, nm))
    return e

sk_count = 0
def subst(e, v, rep):
    if isinstance(e, Var): return rep if e.name == v else e
    if isinstance(e, Func): return Func(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Pred): return Pred(e.name, [subst(a, v, rep) for a in e.args])
    if isinstance(e, Not): return Not(subst(e.op, v, rep))
    if isinstance(e, And): return And(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Or): return Or(subst(e.left, v, rep), subst(e.right, v, rep))
    if isinstance(e, Quant): return Quant(e.qtype, e.var, subst(e.body, v, rep))
    return e

def skolemize(e, uvars=None):
    global sk_count
    uvars = uvars if uvars is not None else []
    if isinstance(e, Quant):
        if e.qtype == "FORALL": return Quant("FORALL", e.var, skolemize(e.body, uvars + [e.var]))
        sk = chr(ord('c') + sk_count) if not uvars else chr(ord('F') + sk_count)
        sk_count += 1
        return skolemize(subst(e.body, e.var, Func(sk, [Var(v) for v in uvars])), uvars)
    if isinstance(e, Not): return Not(skolemize(e.op, uvars))
    if isinstance(e, And): return And(skolemize(e.left, uvars), skolemize(e.right, uvars))
    if isinstance(e, Or): return Or(skolemize(e.left, uvars), skolemize(e.right, uvars))
    return e

def drop_univ(e):
    if isinstance(e, Quant): return drop_univ(e.body)
    if isinstance(e, Not): return Not(drop_univ(e.op))
    if isinstance(e, And): return And(drop_univ(e.left), drop_univ(e.right))
    if isinstance(e, Or): return Or(drop_univ(e.left), drop_univ(e.right))
    return e

def dist_or(e):
    if isinstance(e, And): return And(dist_or(e.left), dist_or(e.right))
    if isinstance(e, Or):
        a, b = dist_or(e.left), dist_or(e.right)
        if isinstance(a, And): return And(dist_or(Or(a.left, b)), dist_or(Or(a.right, b)))
        if isinstance(b, And): return And(dist_or(Or(a, b.left)), dist_or(Or(a, b.right)))
        return Or(a, b)
    return e

def flatten_and(e):
    return flatten_and(e.left) + flatten_and(e.right) if isinstance(e, And) else [str(e)]

if __name__ == "__main__":
    s = input("Enter FOL formula: ").strip()
    e = Parser(tokenize(s)).parse()
    e = elim_imp(e)
    print("1. Eliminate Implications:", e)
    e = move_not(e)
    print("2. Move Negations Inward:", e)
    e = std_vars(e)
    print("3. Standardize Variables:", e)
    e = skolemize(e)
    print("4. Skolemization:", e)
    e = drop_univ(e)
    print("5. Drop Universal Quantifiers:", e)
    e = dist_or(e)
    print("6. Conjunctive Normal Form:", " ∧ ".join(flatten_and(e)))


Enter FOL formula: ~(forall x)(exists y)[~p(x) -> ~q(x,y)] & (exists x)(forall y)q(x,y)
1. Eliminate Implications: (¬((∀x)(∃y)(¬¬p(x) ∨ ¬q(x,y))) ∧ (∃x)(∀y)q(x,y))
2. Move Negations Inward: ((∃x)(∀y)(¬p(x) ∧ q(x,y)) ∧ (∃x)(∀y)q(x,y))
3. Standardize Variables: ((∃x)(∀y)(¬p(x) ∧ q(x,y)) ∧ (∃z)(∀w)q(z,w))
4. Skolemization: ((∀y)(¬p(c) ∧ q(c,y)) ∧ (∀w)q(d,w))
5. Drop Universal Quantifiers: ((¬p(c) ∧ q(c,y)) ∧ q(d,w))
6. Conjunctive Normal Form: ¬p(c) ∧ q(c,y) ∧ q(d,w)
